# 🛣️ Diseño de Pavimentos Flexibles — Método AASHTO 93

---

> **Autor:** Mario Cesgo Soliz  
**Norma de referencia:** AASHTO Guide for Design of Pavement Structures (1993)  
**Versión:** 1.0

---

## Descripción

Este notebook implementa el método **AASHTO 93** para el diseño de pavimentos flexibles,
ampliamente utilizado en Latinoamérica y el mundo. El método es empírico-mecanístico y se
basa en la ecuación de desempeño desarrollada a partir del experimento AASHO Road Test (Ottawa,
Illinois, 1958–1960).

### Contenido del Notebook

1. 📐 Fundamentos teóricos y ecuación de diseño  
2. 📊 Parámetros de entrada  
3. 🔢 Cálculo del Número Estructural (SN)  
4. 📏 Determinación de espesores de capas  
5. 📈 Análisis de sensibilidad  
6. 📋 Memoria de cálculo y resumen  


## 1. Importación de Librerías

In [ ]:


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.35,
})
print("✅ Librerías cargadas correctamente.")


ModuleNotFoundError: No module named 'scipy'

## 2. Fundamentos Teóricos

### 2.1 Ecuación Fundamental de Diseño AASHTO 93

La ecuación de diseño relaciona el número de ejes equivalentes (ESALs) con el
**Número Estructural (SN)** y las condiciones del pavimento:

$$
\log_{10}(W_{18}) = Z_R \cdot S_0 + 9.36 \cdot \log_{10}(SN+1) - 0.20
+ \frac{\log_{10}\!\left(\dfrac{\Delta PSI}{4.2 - 1.5}\right)}{0.40 + \dfrac{1094}{(SN+1)^{5.19}}}
+ 2.32 \cdot \log_{10}(M_R) - 8.07
$$

Donde:
| Símbolo | Descripción | Unidades |
|---------|-------------|----------|
| $W_{18}$ | Número de ejes equivalentes de 18 kip (ESALs) | ejes |
| $Z_R$ | Desviación normal estándar para nivel de confiabilidad $R$ | — |
| $S_0$ | Error estándar combinado | — |
| $SN$ | Número Estructural | pulgadas |
| $\Delta PSI$ | Pérdida de serviciabilidad ($PSI_i - PSI_f$) | — |
| $M_R$ | Módulo de resiliencia de la subrasante | psi |

---

### 2.2 Número Estructural (SN)

El SN se obtiene como la combinación ponderada de los espesores de cada capa:

$$
SN = a_1 \cdot D_1 + a_2 \cdot D_2 \cdot m_2 + a_3 \cdot D_3 \cdot m_3
$$

| Variable | Descripción |
|----------|-------------|
| $a_i$ | Coeficiente estructural de la capa $i$ |
| $D_i$ | Espesor de la capa $i$ (pulgadas) |
| $m_i$ | Coeficiente de drenaje de la capa $i$ |

---

### 2.3 Confiabilidad y Desviación Normal

| Confiabilidad $R$ (%) | $Z_R$ |
|----------------------|-------|
| 50 | 0.000 |
| 75 | −0.674 |
| 90 | −1.282 |
| 95 | −1.645 |
| 99 | −2.327 |


## 3. Parámetros de Diseño

In [ ]:
# ══════════════════════════════════════════════════════════
#  PARÁMETROS DE ENTRADA — Modifique según su proyecto
# ══════════════════════════════════════════════════════════

# ── Tráfico ──────────────────────────────────────────────
W18 = 5_000_000          # ESALs (ejes equivalentes de 18 kip)

# ── Confiabilidad ────────────────────────────────────────
R   = 95                 # Confiabilidad (%)  opciones: 50,75,80,85,90,95,99
ZR_tabla = {50: 0.000, 75: -0.674, 80: -0.842, 85: -1.037,
            90: -1.282, 95: -1.645, 99: -2.327}
ZR  = ZR_tabla[R]

S0  = 0.45               # Error estándar combinado (típico 0.40–0.50)

# ── Serviciabilidad ──────────────────────────────────────
PSI_i = 4.2              # Índice de servicio inicial
PSI_f = 2.5              # Índice de servicio final (vías primarias: 2.5, secundarias: 2.0)
DPSI  = PSI_i - PSI_f

# ── Subrasante ───────────────────────────────────────────
CBR_sub = 8.0            # CBR de la subrasante (%)
# Conversión CBR → Mr (psi) — fórmula AASHTO para suelos finos
MR = 1500 * CBR_sub      # psi  (alternativa: 2555 * CBR^0.64)

# ── Coeficientes estructurales de capas ─────────────────
# Capa 1: Concreto Asfáltico (CA)
a1 = 0.44               # coef. estructural  (típico 0.40–0.44)

# Capa 2: Base granular
a2 = 0.14               # coef. estructural  (típico 0.10–0.14)
m2 = 1.00               # coef. drenaje      (tabla AASHTO)

# Capa 3: Subbase granular
a3 = 0.11               # coef. estructural  (típico 0.08–0.11)
m3 = 1.00               # coef. drenaje

# ── Espesores mínimos (pulgadas) ─────────────────────────
D1_min = 3.0            # CA mínimo
D2_min = 4.0            # Base mínima
D3_min = 4.0            # Subbase mínima

# ══════════════════════════════════════════════════════════
print("=" * 55)
print("        PARÁMETROS DE DISEÑO — AASHTO 93")
print("=" * 55)
print(f"  ESALs de diseño (W18)         : {W18:>15,.0f} ejes")
print(f"  Confiabilidad (R)             : {R:>15} %")
print(f"  Desviación normal (ZR)        : {ZR:>15.3f}")
print(f"  Error estándar (S0)           : {S0:>15.2f}")
print(f"  PSI inicial                   : {PSI_i:>15.1f}")
print(f"  PSI final                     : {PSI_f:>15.1f}")
print(f"  ΔPSI                          : {DPSI:>15.1f}")
print(f"  CBR subrasante                : {CBR_sub:>15.1f} %")
print(f"  Módulo resiliente Mr          : {MR:>15,.0f} psi")
print("-" * 55)
print(f"  a1 (CA)   = {a1}  |  a2 (Base) = {a2}, m2 = {m2}  |  a3 (SB) = {a3}, m3 = {m3}")
print("=" * 55)


## 4. Implementación de la Ecuación AASHTO 93

In [ ]:
def log_W18_aashto93(SN, ZR, S0, DPSI, MR):
    # Calcula log10(W18) dado el Numero Estructural SN.
    # Ecuacion AASHTO 93 para pavimentos flexibles.
    termino1 = ZR * S0
    termino2 = 9.36 * np.log10(SN + 1) - 0.20
    termino3 = np.log10(DPSI / (4.2 - 1.5)) / (0.40 + 1094 / (SN + 1)**5.19)
    termino4 = 2.32 * np.log10(MR) - 8.07
    return termino1 + termino2 + termino3 + termino4


def calcular_SN(W18, ZR, S0, DPSI, MR):
    # Resuelve la ecuacion AASHTO 93 para obtener el SN requerido.
    # Utiliza el metodo de Brent (biseccion robusta).
    objetivo = np.log10(W18)
    f = lambda sn: log_W18_aashto93(sn, ZR, S0, DPSI, MR) - objetivo
    SN_req = brentq(f, 0.5, 20.0, xtol=1e-6)
    return SN_req


# -- Calculo del SN requerido
SN_req = calcular_SN(W18, ZR, S0, DPSI, MR)

print(f"  Numero Estructural requerido  SN = {SN_req:.4f}  pulgadas")
print(f"  Equivalente en cm             SN = {SN_req * 2.54:.2f}  cm (referencial)")


## 5. Curva de Diseño AASHTO 93

In [ ]:
# Curva W18 vs SN para las condiciones del proyecto
SN_range = np.linspace(0.5, 10, 400)
logW18_range = [log_W18_aashto93(sn, ZR, S0, DPSI, MR) for sn in SN_range]
W18_range    = [10**lw for lw in logW18_range]

fig, ax = plt.subplots(figsize=(9, 5))

ax.semilogy(SN_range, W18_range, color='#1a6faf', lw=2.5, label='Curva AASHTO 93')

# Punto de diseño
ax.axvline(SN_req, color='crimson', ls='--', lw=1.5, label=f'SN requerido = {SN_req:.2f}')
ax.axhline(W18,    color='darkorange', ls='--', lw=1.5, label=f'W18 = {W18:,.0f} ESALs')
ax.plot(SN_req, W18, 'o', color='crimson', ms=9, zorder=5)
ax.annotate(f'  SN = {SN_req:.2f}\n  W18 = {W18/1e6:.1f}M',
            xy=(SN_req, W18), xytext=(SN_req+0.5, W18*1.5),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9, color='darkred')

ax.set_xlabel('Número Estructural (SN) [pulg]')
ax.set_ylabel('Número de Ejes Equivalentes W18')
ax.set_title('Curva de Diseño AASHTO 93 — Pavimento Flexible')
ax.legend()
ax.set_xlim(0.5, 10)
plt.tight_layout()
plt.show()


## 6. Determinación de Espesores de Capas

El procedimiento de capas se realiza de forma **secuencial** (de arriba hacia abajo):

1. Determinar $D_1$ tal que $SN_1 \geq SN_{1,\text{req}}$  
2. Determinar $D_2$ tal que $SN_1 + SN_2 \geq SN_{2,\text{req}}$  
3. Determinar $D_3$ tal que $SN_1 + SN_2 + SN_3 \geq SN_{3,\text{req}}$

Los $SN$ requeridos sobre cada capa se calculan para el módulo de la capa inferior:


In [ ]:
# ── Módulos de las capas (psi) ───────────────────────────
# Se usan para calcular el SN requerido sobre cada interfaz
E1 = 450_000    # Módulo CA (psi)   — referencial
E2 = 30_000     # Módulo Base (psi)
E3 = 15_000     # Módulo Subbase (psi)
# MR ya está definido para la subrasante

# SN requeridos sobre cada capa (interface)
# Se calcula el SN para el módulo de la capa de soporte en cada nivel
SN_sobre_base    = calcular_SN(W18, ZR, S0, DPSI, E2)
SN_sobre_subbase = calcular_SN(W18, ZR, S0, DPSI, E3)
SN_total         = SN_req  # ya calculado sobre la subrasante

print(f"  SN requerido sobre Base    (E2={E2:,} psi) : {SN_sobre_base:.3f} pulg")
print(f"  SN requerido sobre Subbase (E3={E3:,} psi) : {SN_sobre_subbase:.3f} pulg")
print(f"  SN requerido total         (MR={MR:,} psi) : {SN_total:.3f} pulg")
print()

# ── Capa 1: Concreto Asfáltico ───────────────────────────
D1_calc = SN_sobre_base / a1
D1 = max(D1_calc, D1_min)
D1 = np.ceil(D1 / 0.5) * 0.5          # redondear a 0.5 pulg
SN1 = a1 * D1

# ── Capa 2: Base granular ────────────────────────────────
SN2_req = SN_sobre_subbase - SN1
D2_calc = SN2_req / (a2 * m2) if SN2_req > 0 else D2_min
D2 = max(D2_calc, D2_min)
D2 = np.ceil(D2 / 0.5) * 0.5
SN2 = a2 * m2 * D2

# ── Capa 3: Subbase granular ─────────────────────────────
SN3_req = SN_total - SN1 - SN2
D3_calc = SN3_req / (a3 * m3) if SN3_req > 0 else D3_min
D3 = max(D3_calc, D3_min)
D3 = np.ceil(D3 / 0.5) * 0.5
SN3 = a3 * m3 * D3

SN_prov = SN1 + SN2 + SN3

# ── Conversión a cm ──────────────────────────────────────
D1_cm = D1 * 2.54
D2_cm = D2 * 2.54
D3_cm = D3 * 2.54

print("=" * 55)
print("        DISEÑO DE ESPESORES — AASHTO 93")
print("=" * 55)
print(f"  Capa 1 – Concreto Asfáltico")
print(f"    D1 = {D1:.1f} pulg  →  {D1_cm:.1f} cm   (SN1 = {SN1:.3f})")
print(f"  Capa 2 – Base Granular")
print(f"    D2 = {D2:.1f} pulg  →  {D2_cm:.1f} cm   (SN2 = {SN2:.3f})")
print(f"  Capa 3 – Subbase Granular")
print(f"    D3 = {D3:.1f} pulg  →  {D3_cm:.1f} cm   (SN3 = {SN3:.3f})")
print("-" * 55)
print(f"  SN provisto  = {SN_prov:.3f}  pulg")
print(f"  SN requerido = {SN_total:.3f}  pulg")
ok = "✅ OK" if SN_prov >= SN_total else "❌ INSUFICIENTE"
print(f"  Verificación : {ok}")
print("=" * 55)


## 7. Sección Transversal Típica

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.set_xlim(0, 10)
ax.set_aspect('equal')
ax.axis('off')

capas = [
    (D1_cm, '#2c2c2c', f'Concreto Asfáltico\nD1 = {D1_cm:.1f} cm  ({D1:.1f}")', 'white'),
    (D2_cm, '#b5651d', f'Base Granular\nD2 = {D2_cm:.1f} cm  ({D2:.1f}")', 'white'),
    (D3_cm, '#c8a96e', f'Subbase Granular\nD3 = {D3_cm:.1f} cm  ({D3:.1f}")', '#3a2000'),
    (30,    '#8b7355', 'Subrasante\n(compactada)', '#1a0a00'),
]

y = 0
total_h = sum(c[0] for c in capas)
scale = 8 / total_h   # escala para dibujo

for (h, color, label, fcolor) in capas:
    hd = h * scale
    rect = mpatches.FancyBboxPatch((1, y), 8, hd,
                                   boxstyle="square,pad=0",
                                   facecolor=color, edgecolor='white', lw=1.5)
    ax.add_patch(rect)
    ax.text(5, y + hd/2, label, ha='center', va='center',
            fontsize=10, color=fcolor, fontweight='bold', linespacing=1.5)
    y += hd

# Flechas de cotas
y = 0
for (h, color, label, fcolor) in capas[:3]:
    hd = h * scale
    ax.annotate('', xy=(0.5, y + hd), xytext=(0.5, y),
                arrowprops=dict(arrowstyle='<->', color='#333', lw=1.2))
    ax.text(0.35, y + hd/2, f'{h:.1f} cm', ha='right', va='center', fontsize=8.5)
    y += hd

ax.set_title(f'Sección Típica de Pavimento Flexible — AASHTO 93\n'
             f'SN provisto = {SN_prov:.2f}  |  W18 = {W18/1e6:.1f}M ESALs  |  R = {R}%',
             fontsize=12, pad=12)
plt.tight_layout()
plt.show()


## 8. Análisis de Sensibilidad

In [ ]:
# ── 8.1 Sensibilidad al CBR de la subrasante ─────────────
CBR_vals = np.linspace(2, 20, 100)
MR_vals  = 1500 * CBR_vals
SN_vals  = [calcular_SN(W18, ZR, S0, DPSI, mr) for mr in MR_vals]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico izquierdo: SN vs CBR
ax = axes[0]
ax.plot(CBR_vals, SN_vals, color='#1a6faf', lw=2.5)
ax.axvline(CBR_sub, color='crimson', ls='--', lw=1.5, label=f'CBR proyecto = {CBR_sub}%')
ax.axhline(SN_req,  color='darkorange', ls='--', lw=1.5, label=f'SN = {SN_req:.2f}')
ax.plot(CBR_sub, SN_req, 'o', color='crimson', ms=9)
ax.set_xlabel('CBR de Subrasante (%)')
ax.set_ylabel('Número Estructural SN [pulg]')
ax.set_title('Sensibilidad: SN vs CBR')
ax.legend()

# Gráfico derecho: Espesor total vs CBR
espesores_total = []
for mr in MR_vals:
    sn = calcular_SN(W18, ZR, S0, DPSI, mr)
    d1 = max(np.ceil((sn / a1) / 0.5) * 0.5, D1_min)
    sn1 = a1 * d1
    sn2_req = max(sn - sn1, 0)
    d2 = max(np.ceil((sn2_req / (a2*m2)) / 0.5) * 0.5, D2_min)
    sn2 = a2 * m2 * d2
    sn3_req = max(sn - sn1 - sn2, 0)
    d3 = max(np.ceil((sn3_req / (a3*m3)) / 0.5) * 0.5, D3_min)
    espesores_total.append((d1 + d2 + d3) * 2.54)

ax = axes[1]
ax.plot(CBR_vals, espesores_total, color='#2ca02c', lw=2.5)
ax.axvline(CBR_sub, color='crimson', ls='--', lw=1.5, label=f'CBR = {CBR_sub}%')
ax.set_xlabel('CBR de Subrasante (%)')
ax.set_ylabel('Espesor Total del Paquete [cm]')
ax.set_title('Espesor Total vs CBR')
ax.legend()

plt.suptitle('Análisis de Sensibilidad — Método AASHTO 93', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── 8.2 Sensibilidad al Tráfico (ESALs) ─────────────────
W18_range2 = np.logspace(5, 8, 200)
SN_traf    = [calcular_SN(w, ZR, S0, DPSI, MR) for w in W18_range2]

# ── 8.3 Sensibilidad a la Confiabilidad ──────────────────
conf_vals = [50, 75, 80, 85, 90, 95, 99]
colores   = ['#aec7e8','#6baed6','#4292c6','#2171b5','#08519c','#08306b','#03163a']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.semilogx(W18_range2, SN_traf, color='#9467bd', lw=2.5)
ax.axvline(W18, color='crimson', ls='--', lw=1.5, label=f'W18 diseño = {W18:,.0f}')
ax.axhline(SN_req, color='darkorange', ls='--', lw=1.5, label=f'SN = {SN_req:.2f}')
ax.plot(W18, SN_req, 'o', color='crimson', ms=9)
ax.set_xlabel('W18 — Número de Ejes Equivalentes')
ax.set_ylabel('Número Estructural SN [pulg]')
ax.set_title('Sensibilidad: SN vs Tráfico')
ax.legend()

ax = axes[1]
for c, col in zip(conf_vals, colores):
    zr_c = ZR_tabla[c]
    SN_c = [calcular_SN(w, zr_c, S0, DPSI, MR) for w in W18_range2]
    ax.semilogx(W18_range2, SN_c, color=col, lw=1.8, label=f'R={c}%')

ax.set_xlabel('W18 — Número de Ejes Equivalentes')
ax.set_ylabel('Número Estructural SN [pulg]')
ax.set_title('SN vs Tráfico para distintas Confiabilidades')
ax.legend(fontsize=8)

plt.suptitle('Análisis de Sensibilidad — Tráfico y Confiabilidad', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 9. Coeficientes de Drenaje

Los coeficientes $m_i$ ajustan el aporte estructural según la calidad del drenaje
y el porcentaje del tiempo en que la estructura está expuesta a niveles de humedad
próximos a la saturación.

| Calidad del Drenaje | Tiempo para drenar | % Tiempo cercano a saturación | $m_i$ |
|---------------------|-------------------|-------------------------------|-------|
| Excelente | < 2 horas | 1% | 1.40–1.35 |
| Bueno | < 1 día | 5% | 1.35–1.25 |
| Regular | < 1 semana | 25% | 1.25–1.15 |
| Pobre | < 1 mes | 25% | 1.15–1.05 |
| Muy pobre | No drena | 25% | 1.05–0.95 |


In [ ]:
# Tabla de coeficientes de drenaje para visualización
drenaje = {
    'Excelente': [1.40, 1.35, 1.30, 1.20],
    'Bueno':     [1.35, 1.25, 1.15, 1.00],
    'Regular':   [1.25, 1.15, 1.05, 0.80],
    'Pobre':     [1.15, 1.05, 0.80, 0.60],
    'Muy Pobre': [1.05, 0.95, 0.75, 0.40],
}
pct_sat = ['< 1%', '1–5%', '5–25%', '> 25%']

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(pct_sat))
colores2 = ['#2ca02c','#1f77b4','#ff7f0e','#d62728','#9467bd']
for i, (tipo, vals) in enumerate(drenaje.items()):
    ax.plot(x, vals, 'o-', color=colores2[i], lw=2, ms=7, label=tipo)

ax.set_xticks(x)
ax.set_xticklabels([f'% sat: {p}' for p in pct_sat])
ax.set_ylabel('Coeficiente de Drenaje $m_i$')
ax.set_title('Coeficientes de Drenaje AASHTO 93')
ax.legend(title='Calidad del drenaje', fontsize=9)
ax.set_ylim(0.3, 1.6)
plt.tight_layout()
plt.show()


## 10. Cálculo del Tráfico de Diseño (ESALs)

El número de ejes equivalentes $W_{18}$ se estima a partir del tráfico diario por tipo de vehículo:

$$
W_{18} = \sum_j \text{TPDA}_j \cdot f_{c,j} \cdot 365 \cdot n \cdot F_D \cdot F_C
$$

Donde $f_{c,j}$ es el **Factor Equivalente de Carga (FEC)** del vehículo $j$,
y $n$ es el período de diseño en años.

El FEC por eje se calcula como:
$$
\text{FEC} = \left(\frac{P}{P_{\text{ref}}}\right)^4
\quad \text{con } P_{\text{ref}} = 8.2 \text{ ton (eje sencillo)}
$$


In [ ]:
# ── Definición de flota vehicular ───────────────────────
vehiculos = {
    'Bus':            {'TPDA': 200,  'FEC': 1.20, 'fc_dir': 0.5, 'fc_carr': 0.85},
    'Camión C2':      {'TPDA': 300,  'FEC': 2.45, 'fc_dir': 0.5, 'fc_carr': 0.85},
    'Camión C3':      {'TPDA': 150,  'FEC': 3.80, 'fc_dir': 0.5, 'fc_carr': 0.85},
    'Semi-tráiler':   {'TPDA':  80,  'FEC': 5.20, 'fc_dir': 0.5, 'fc_carr': 0.85},
}

n_anios = 20          # Período de diseño (años)
tasa_crecimiento = 0.03  # Tasa de crecimiento del tráfico (3%)

# Factor de crecimiento acumulado
FCR = ((1 + tasa_crecimiento)**n_anios - 1) / tasa_crecimiento

print(f"  Período de diseño : {n_anios} años")
print(f"  Tasa de crecimiento : {tasa_crecimiento*100:.1f}%")
print(f"  Factor de crecimiento acumulado (FCR) : {FCR:.2f}")
print()
print(f"  {'Vehículo':<18} {'TPDA':>6} {'FEC':>6} {'ESALs/año':>12} {'ESALs total':>14}")
print("  " + "-"*60)

W18_total = 0
esal_anual = {}
for veh, datos in vehiculos.items():
    ea = datos['TPDA'] * datos['FEC'] * 365 * datos['fc_dir'] * datos['fc_carr']
    et = ea * FCR
    W18_total += et
    esal_anual[veh] = ea
    print(f"  {veh:<18} {datos['TPDA']:>6,.0f} {datos['FEC']:>6.2f} {ea:>12,.0f} {et:>14,.0f}")

print("  " + "-"*60)
print(f"  {'TOTAL':<18} {'':>6} {'':>6} {'':>12} {W18_total:>14,.0f}")
print()
print(f"  ➡  W18 de diseño = {W18_total:,.0f} ESALs")

# ── Gráfico de composición del tráfico ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
etiquetas = list(vehiculos.keys())
TPDAs = [v['TPDA'] for v in vehiculos.values()]
ax.bar(etiquetas, TPDAs, color=['#4e79a7','#f28e2b','#e15759','#76b7b2'])
ax.set_ylabel('TPDA (vehículos/día)')
ax.set_title('Tráfico Promedio Diario Anual por Tipo')
ax.tick_params(axis='x', rotation=15)

ax = axes[1]
ESALs_v = [vehiculos[v]['TPDA'] * vehiculos[v]['FEC'] * 365 *
           vehiculos[v]['fc_dir'] * vehiculos[v]['fc_carr'] * FCR
           for v in vehiculos]
wedges, texts, autotexts = ax.pie(ESALs_v, labels=etiquetas,
    autopct='%1.1f%%', colors=['#4e79a7','#f28e2b','#e15759','#76b7b2'],
    startangle=90, pctdistance=0.75)
ax.set_title('Distribución de ESALs por Tipo de Vehículo')

plt.tight_layout()
plt.show()


## 11. Memoria de Cálculo y Resumen Final

In [ ]:
print("=" * 65)
print("       MEMORIA DE CÁLCULO — DISEÑO AASHTO 93")
print("       PAVIMENTO FLEXIBLE")
print("=" * 65)
print()
print("  1. PARÁMETROS DE DISEÑO")
print(f"     W18 (ESALs)         = {W18:>15,.0f}")
print(f"     Confiabilidad R     = {R:>14} %")
print(f"     ZR                  = {ZR:>15.3f}")
print(f"     S0                  = {S0:>15.2f}")
print(f"     PSI inicial         = {PSI_i:>15.1f}")
print(f"     PSI final           = {PSI_f:>15.1f}")
print(f"     ΔPSI                = {DPSI:>15.1f}")
print(f"     CBR subrasante      = {CBR_sub:>14.1f} %")
print(f"     Mr subrasante       = {MR:>12,.0f} psi")
print()
print("  2. COEFICIENTES ESTRUCTURALES Y DRENAJE")
print(f"     a1 (CA)             = {a1:>15.2f}")
print(f"     a2 (Base), m2       = {a2:>8.2f}, m2 = {m2}")
print(f"     a3 (Subbase), m3    = {a3:>8.2f}, m3 = {m3}")
print()
print("  3. NÚMERO ESTRUCTURAL REQUERIDO")
print(f"     SN requerido        = {SN_req:>15.4f}  pulg")
print()
print("  4. ESPESORES DE DISEÑO")
print(f"     D1 (CA)             = {D1:>6.1f} pulg  =  {D1_cm:>6.1f} cm")
print(f"     D2 (Base)           = {D2:>6.1f} pulg  =  {D2_cm:>6.1f} cm")
print(f"     D3 (Subbase)        = {D3:>6.1f} pulg  =  {D3_cm:>6.1f} cm")
print(f"     Total               = {(D1+D2+D3):>6.1f} pulg  =  {(D1_cm+D2_cm+D3_cm):>6.1f} cm")
print()
print("  5. VERIFICACIÓN ESTRUCTURAL")
print(f"     SN1 = {SN1:.3f}  |  SN2 = {SN2:.3f}  |  SN3 = {SN3:.3f}")
print(f"     SN provisto  = {SN_prov:.4f}  pulg")
print(f"     SN requerido = {SN_total:.4f}  pulg")
ok = "✅ DISEÑO SATISFACTORIO" if SN_prov >= SN_total else "❌ DISEÑO INSUFICIENTE"
print(f"     {ok}")
print("=" * 65)



---
> **Nota:** Este programa es una herramienta de apoyo al diseño. Los resultados deben
> ser verificados por un ingeniero especialista.
